# Response-Surface Optimization of a Manufacturing Process with PROC ORTHOREG

## Executive Summary

A process engineer runs a two-factor central composite design (CCD) to study how reactor **temperature** and **pressure** drive product **yield**, then fits a full second-order response-surface model with PROC ORTHOREG. ORTHOREG's Householder-based least squares delivers numerically stable estimates even when the squared and cross-product terms make the design matrix ill-conditioned; the fitted surface is then searched on a grid to locate the optimal operating set point.

## Data Sources

| Dataset | Rows | Description | Key variables |
|---------|------|-------------|---------------|
| `ccd` | 75 | Synthetic two-factor central composite design (5 coded levels per factor x 3 replicates) for a chemical reactor, generated inline with `call streaminit`/`rand`. | `temp_c`, `press_c` (coded factor levels in [-1.414, 1.414]), `temperature` (deg C), `pressure` (bar), `yield` (% conversion response), `rep` (replicate id) |
| `rsm_est` | 1 | OUTEST= output from PROC ORTHOREG holding the fitted second-order coefficients. | `_TYPE_` (=`PARMS`), `_NAME_` (blank), `_RMSE_`, `Intercept`, `Col1`-`Col5` (one per MODEL effect) |
| `grid` | 1 | DATA-step grid search over the fitted surface to find the predicted optimum. | `best` (max predicted yield), `bt_opt`, `bp_opt` (optimal coded settings) |

# Response-Surface Optimization with PROC ORTHOREG

A process engineering team wants to maximize the percentage **yield** of a chemical reactor by tuning two controllable factors: **temperature** and **pressure**. Rather than change one factor at a time, they run a **central composite design (CCD)** — the workhorse design of response-surface methodology (RSM) — and fit a full second-order (quadratic) model.

The second-order model includes pure quadratic terms (`temp*temp`, `press*press`) and a cross-product (`temp*press`). These terms are strongly correlated with the linear terms, which makes the design matrix **ill-conditioned**. PROC ORTHOREG is built for exactly this situation: it solves the least-squares problem with Householder transformations and extended-precision arithmetic, so the coefficient estimates stay accurate where the normal-equations route used by ordinary regression can lose digits.

## Step 1 - Generate the experimental design

We simulate a textbook **rotatable CCD** for two factors. Each factor takes five coded levels — the factorial points (-1, +1), the axial/star points (+/- 1.414), and the center point (0) — giving a 5x5 grid that we replicate three times to estimate pure error.

The "true" process is a known quadratic surface (an interior maximum, because both quadratic coefficients are negative). We add Gaussian measurement noise so the fitted model has to recover the signal from realistic, noisy runs. Coded factors are also mapped back to engineering units (deg C and bar) for interpretability.

In [1]:
data ccd;
   call streaminit(20260531);

   /* Five coded levels of a rotatable 2-factor CCD:
      axial (-1.414), factorial (-1, +1), center (0), axial (+1.414) */
   array lev[5] _temporary_ (-1.41421 -1 0 1 1.41421);

   /* Known "true" response surface (interior maximum) */
   b0  = 78.5;   /* baseline yield at the center point   */
   bt  =  4.2;   /* linear effect of temperature          */
   bp  =  2.7;   /* linear effect of pressure             */
   btt = -3.1;   /* quadratic (curvature) in temperature  */
   bpp = -2.4;   /* quadratic (curvature) in pressure     */
   btp =  1.6;   /* temperature * pressure interaction    */

   do rep = 1 to 3;
      do ti = 1 to 5;
         do pi = 1 to 5;
            temp_c  = lev[ti];
            press_c = lev[pi];

            /* Mean response on the true quadratic surface */
            mu = b0
               + bt*temp_c + bp*press_c
               + btt*temp_c*temp_c
               + bpp*press_c*press_c
               + btp*temp_c*press_c;

            /* Observed yield = surface + run-to-run noise */
            yield = mu + rand('normal', 0, 0.8);

            /* Coded levels mapped back to engineering units */
            temperature = 160 + 10*temp_c;   /* deg C */
            pressure    =  30 +  5*press_c;   /* bar   */

            output;
         end;
      end;
   end;

   keep rep temperature pressure temp_c press_c yield;
run;

NOTE: DATA ccd


NOTE: Wrote ccd (75 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Step 2 - Inspect the design and response

Before modeling, confirm the design is balanced and centered: the coded factors should average to 0 and span the [-1.414, +1.414] range. A quick look at the yield range tells us how much the response moves across the design space.

In [2]:
proc means data=ccd n mean min max maxdec=4;
   var yield temp_c press_c;
   title 'Central Composite Design: Factor Levels and Yield Range';
run;

                                Central Composite Design: Factor Levels and Yield Range                                 

                                                  The MEANS Procedure

 Variable        N           Mean     Minimum     Maximum
 --------------------------------------------------------
 yield          75        71.9311     60.4556     81.9495
 temp_c         75        -0.0000     -1.4142      1.4142
 press_c        75         0.0000     -1.4142      1.4142
 --------------------------------------------------------



NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Step 3 - Fit the second-order response surface

The full quadratic model regresses `yield` on the two linear terms, the two pure quadratic terms, and the cross-product. PROC ORTHOREG fits it with numerically stable Householder least squares.

The `OUTEST=` option saves the estimated coefficients to a dataset so we can reuse them downstream to predict and optimize. ORTHOREG also prints a parameter-estimates table and overall fit statistics (R-square, Root MSE) by default.

In [3]:
proc orthoreg data=ccd outest=rsm_est;
   model yield = temp_c press_c
                 temp_c*temp_c
                 press_c*press_c
                 temp_c*press_c;
   title 'Second-Order Response Surface for Reactor Yield';
run;

                                Central Composite Design: Factor Levels and Yield Range                                 

  The ORTHOREG Procedure  
Dependent Variable: yield 

Parameter         Estimate
---------------  ---------
Intercept        78.267924
temp_c            4.266253
press_c           2.604904
temp_c*temp_c    -2.882002
press_c*press_c   -2.39874
temp_c*press_c    1.575276

  Fit Statistics   

Statistic     Value
---------  --------
R-Square   0.989149
Root MSE    0.70287



NOTE: PROC ORTHOREG data=ccd

NOTE: Wrote rsm_est (1 row(s), 9 columns).


## Step 4 - Review the saved coefficients

The `OUTEST=` dataset follows the SAS regression-procedure convention. It opens with the special variables `_TYPE_` (here `PARMS`, flagging the row as parameter estimates) and `_NAME_` (blank for an ordinary single-response fit), then carries the Root MSE (`_RMSE_`), the `Intercept`, and one column per MODEL effect (`Col1`-`Col5`, in the order the terms appear in the MODEL statement). Printing the single `PARMS` row confirms the recovered surface and gives us the numbers we need for prediction.

In [4]:
proc print data=rsm_est noobs;
   title 'Fitted Response-Surface Coefficients (OUTEST=)';
run;

                                     Fitted Response-Surface Coefficients (OUTEST=)                                     

_TYPE_  _NAME_        _RMSE_      INTERCEPT          COL1          COL2           COL3           COL4          COL5
PARMS           0.7028698526  78.2679237381  4.2662533021  2.6049036312  -2.8820023675  -2.3987400762  1.5752758873



NOTE: PROC PRINT data=rsm_est

NOTE: PROC PRINT completed: 1 observations printed, 9 variables


## Step 5 - Locate the optimal operating point

With a fitted quadratic surface in hand, the engineering question is: *which temperature and pressure maximize yield?* We read the saved coefficients into a DATA step and evaluate the predicted surface on a fine grid of coded settings, keeping the point with the highest predicted yield.

Because both quadratic coefficients are negative, the surface is concave and has an interior maximum — the grid search converges on the predicted sweet spot.

In [5]:
data grid;
   /* Pull the fitted coefficients in on the first iteration */
   if _n_ = 1 then set rsm_est;
   retain b0 bt bp btt bpp btp;
   if _n_ = 1 then do;
      b0  = intercept;
      bt  = col1;   /* temp_c          */
      bp  = col2;   /* press_c         */
      btt = col3;   /* temp_c*temp_c   */
      bpp = col4;   /* press_c*press_c */
      btp = col5;   /* temp_c*press_c  */
   end;

   /* Search the fitted surface over the coded design region */
   best = .;
   do t = -1.5 to 1.5 by 0.05;
      do p = -1.5 to 1.5 by 0.05;
         pred = b0 + bt*t + bp*p
              + btt*t*t + bpp*p*p + btp*t*p;
         if best = . or pred > best then do;
            best   = pred;
            bt_opt = t;
            bp_opt = p;
         end;
      end;
   end;

   /* Translate the optimal coded settings to engineering units */
   temp_opt  = 160 + 10*bt_opt;
   press_opt =  30 +  5*bp_opt;

   keep best bt_opt bp_opt temp_opt press_opt;
   output;
   stop;
run;

proc print data=grid noobs;
   title 'Predicted Optimum Operating Point';
run;

                                           Predicted Optimum Operating Point                                            

         BEST  BT_OPT  BP_OPT  TEMP_OPT  PRESS_OPT
81.4729708988    0.95    0.85     169.5      34.25



NOTE: DATA grid


NOTE: Wrote grid (1 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=grid

NOTE: PROC PRINT completed: 1 observations printed, 5 variables


## Interpreting the results

- **The model fits well.** PROC ORTHOREG recovers the true surface with R-square near 0.99 and a small Root MSE, despite the squared and cross-product terms making the design matrix ill-conditioned. The estimated coefficients land close to the simulated truth (intercept ~78.5, positive linear effects, negative curvature in both factors, positive interaction) — confirming ORTHOREG's numerically stable least squares earns its keep on response-surface designs.
- **The surface is concave with an interior optimum.** Negative quadratic coefficients on both factors mean yield rises, peaks, and falls as each factor increases — exactly the behavior an engineer hopes to find, because it implies a controllable best set point rather than a runaway boundary.
- **The grid search points to the sweet spot.** Reading the saved `OUTEST=` coefficients back into a DATA step turns the abstract regression into an actionable recommendation: run the reactor at the predicted optimal temperature and pressure to maximize yield.

**Why ORTHOREG here?** Response-surface models routinely include quadratic and interaction terms that are highly collinear with the linear terms. On poorly conditioned designs, ordinary normal-equations regression can lose precision in exactly these coefficients — the ones that define the curvature and therefore the location of the optimum. ORTHOREG's Householder/Gentleman-Givens approach preserves those digits, so the optimization built on top of the fit is trustworthy. In a real program you would follow up with confirmation runs at the predicted optimum and, if needed, a steepest-ascent path or a canonical analysis of the fitted surface.